In [1]:
import os
import pandas as pd
from openai import OpenAI
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

# Loading dataset

In [2]:
data = pd.read_csv('https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv', encoding='latin-1')
data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
data.columns = ['label', 'content']
data["label"] = data["label"].map({"spam": 1, "ham": 0})
data.head()

,label,content
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


getting random 500 messages from dataset

In [3]:
sample_data = data.sample(500, random_state=64)
sample_data.head()

,label,content
2415,0,Huh means computational science... Y they like...
3951,0,Me hungry buy some food good lei... But mum n ...
3765,0,Yes.mum lookin strong:)
1303,0,I cant pick the phone right now. Pls send a me...
1623,0,"Ah, well that confuses things, doesnÛ÷t it?"


# OpenAI part

In [4]:
os.environ["OPENAI_API_KEY"] = "A_TYT_NET_TOKENA("

In [5]:
def classify_message_with_openai(client, message):
    messages = [
        {
          "role": "system",
          "content": "You are an AI model that classifies messages into spam or not spam. A spam message is one that is unsolicited, promotional, or deceptive. A non-spam message is normal, conversational, or expected.\n\nExamples:\n\nNon-spam: \"Ok... Ur typical reply...\"\nSpam: \"FreeMsg Why haven't you replied to my text? I'm Randy, sexy, female and live local. Luv to hear from u. Netcollex Ltd 08700621170150p per msg reply Stop to end.\""
        },
        {"role": "user", "content": f"Message: {message}\nSpam: true/false?"}
    ]
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            max_tokens=5,
            temperature=0
        )
        result = response.choices[0].message.content.strip().lower()
        return result == "true"
    except Exception as e:
        print(f"Error processing message: {e}")
        return None


In [6]:
client = OpenAI()
predictions = []
for text in tqdm(sample_data["content"]):
    prediction = classify_message_with_openai(client, text)
    predictions.append(int(prediction) if prediction is not None else 0)

# Добавляем предсказания в DataFrame
sample_data["prediction"] = predictions

100%|████████████████████████████████████████████████████████████████████████████████| 500/500 [05:22<00:00,  1.55it/s]


In [7]:
true_labels = sample_data["label"]
predicted_labels = sample_data["prediction"]
accuracy = accuracy_score(true_labels, predicted_labels)

print("-"*50)
print(classification_report(true_labels, predicted_labels, target_names=["Ham", "Spam"]))
print("-"*50)

--------------------------------------------------
              precision    recall  f1-score   support

         Ham       0.88      1.00      0.94       439
        Spam       0.00      0.00      0.00        61

    accuracy                           0.88       500
   macro avg       0.44      0.50      0.47       500
weighted avg       0.77      0.88      0.82       500

--------------------------------------------------


C:\Users\Ega22\anaconda3\envs\dl_course\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Ega22\anaconda3\envs\dl_course\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Ega22\anaconda3\envs\dl_course\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu